# Our Method PFC dataset training

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Fit the nine-section anatomical model

In [ ]:
# Qian PFC: Spatial TI on ALL 9 independently acquired sections.
# Match the EXACT Stage-1a MODEL/TRAINING DEFAULTS used in the frozen
# synthetic developmental crescent benchmark (dataset seed 20261023).
#
# IMPORTANT:
#   Synthetic DATASET seed = 20261023.
#   Stage-1a TRAINING seed = 20260622 (repository default).
#   They are different for a reason and no synthetic retraining is necessary.
#
# This run uses full PFC RAW counts and the same selected Flow-STP source
# commit. CPU rather than GPU may yield small numerical differences.
# No H1, layer, anatomical annotation, or test label is used for fitting.
#
# Use a fresh Jupyter kernel in a Slurm CPU compute-node allocation.
# Full run: 4,378,919 cells, 9 sections, 300 genes, 1500 epochs.
#
# Output directory deliberately differs from all earlier training runs.
# The learned E(z,t) supports derived Fisher V/q and 128-step RK4 subsequently.

from pathlib import Path
from dataclasses import asdict, replace
import os, sys, json, hashlib, random, subprocess, inspect, time

# CPU must be selected before any torch CUDA initialization.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    AFFINITY_CPUS = len(os.sched_getaffinity(0))
except AttributeError:
    AFFINITY_CPUS = os.cpu_count() or 4
SLURM_CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", AFFINITY_CPUS))
CPU_THREADS = max(1, min(AFFINITY_CPUS, SLURM_CPUS))
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)

import numpy as np
import pandas as pd
import torch

torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(min(CPU_THREADS, 4))
except RuntimeError:
    pass
if torch.cuda.is_available():
    raise RuntimeError(
        "CUDA remains available, likely because torch was already initialized. "
        "Start a fresh CPU-node Jupyter kernel and run this cell first."
    )
assert torch.device("cpu").type == "cpu"
if "SLURM_JOB_ID" not in os.environ:
    print("WARNING: no Slurm allocation found. Do not run the FULL fit on a login node.", flush=True)

# -----------------------------------------------------------------------
# A. Data, code, and new run location
# -----------------------------------------------------------------------
PROJECT = Path(os.getenv(
    "STP_PROJECT",
    os.environ["STP_PROJECT"]
))
DATA_ROOT = Path(os.getenv(
    "QIAN_PFC_DATA_ROOT",
    os.environ["QIAN_PFC_DATA_ROOT"]
))
REPO = Path(os.getenv(
    "FLOW_STP_REPO",
    os.environ["FLOW_STP_REPO"]
))
if not (REPO / "src/flow_stp/stage1/stage1a.py").is_file():
    # Fallback is allowed only if identical source checks below pass.
    if (PROJECT / "src/flow_stp/stage1/stage1a.py").is_file():
        REPO = PROJECT
    else:
        raise FileNotFoundError(f"No Flow-STP checkout at {REPO} or {PROJECT}")

PINNED_COMMIT = "76bb952a841c3671ec35161149e780167fd9b7b1"
EXPECTED_METHOD_VERSION = "stage1a_shared_logE_global_gauge_v1"

SMOKE = False  # True: 3 epochs, 3k cells/section; DIFFERENT smoke-only run.
OUT_ROOT = PROJECT / "experiments/qian_pfc_spatial_ti_all9_synthetic_defaults_v1"
OUTPUT = OUT_ROOT / ("CPU_SMOKE" if SMOKE else "CPU_FULL")
RUN_LOCK = OUTPUT / "locked_config.json"
FIT_DONE = OUTPUT / "FIT_VERIFIED.json"
DYNAMICS_FILE = OUTPUT / "fisher_V_q_grid_public_only.npz"

EXPECTED_PANEL = [
    (0, "GW15", "UMB1117", "F1a", 548567),
    (1, "GW15", "UMB1117", "F1b", 410492),
    (2, "GW20", "FB080", "F1", 665204),
    (3, "GW20", "FB080", "F2a", 596754),
    (4, "GW20", "FB080", "F2b", 322648),
    (5, "GW20", "FB121", "F1", 553738),
    (6, "GW22", "FB123", "F1", 508417),
    (7, "GW22", "FB123", "F2", 439986),
    (8, "GW22", "FB123", "F3", 333113),
]
assert sum(row[4] for row in EXPECTED_PANEL) == 4_378_919

def shasum(file):
    h = hashlib.sha256()
    with open(file, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def git(*args):
    result = subprocess.run(
        ["git", "-C", str(REPO), *args], text=True,
        capture_output=True, check=True
    )
    return result.stdout.strip()

# EXACT executable commit used for synthetic training.
repo_commit = git("rev-parse", "HEAD")
if repo_commit != PINNED_COMMIT:
    raise RuntimeError(
        f"Source commit does not match synthetic benchmark.\n"
        f"  current:   {repo_commit}\n"
        f"  synthetic: {PINNED_COMMIT}\n"
        f"Use the pinned checkout or explicitly start a separately labeled "
        f"cross-version experiment; do not silently call them identical."
    )
dirty = git(
    "status", "--porcelain", "--untracked-files=no", "--", "src/flow_stp"
)
if dirty:
    raise RuntimeError(
        "Tracked Flow-STP source files are modified despite matching git HEAD:\n"
        + dirty
    )

sys.path.insert(0, str(REPO / "src"))
import flow_stp
from flow_stp.data.qian_pfc import (
    QianPFCConfig, build_qian_pfc_panel, load_qian_pfc_for_stage1
)
from flow_stp.stage1.stage1a import (
    METHOD_VERSION, Stage1ARunConfig, FlowSTPConfig, FlowSTPModel, run_stage1a
)

source_path = Path(inspect.getfile(Stage1ARunConfig)).resolve()
if not source_path.is_relative_to(REPO.resolve()):
    raise RuntimeError(
        f"Stale Flow-STP already imported from {source_path}. "
        f"Restart Jupyter kernel and run this cell first."
    )
assert METHOD_VERSION == EXPECTED_METHOD_VERSION, (
    METHOD_VERSION, EXPECTED_METHOD_VERSION
)

# EXACT SAME CONSTRUCTION as attached synthetic notebook:
#   cfg = Stage1ARunConfig(output_dir=NATIVE)
# No hand-picked learning rate, optimizer parameters or batch size.
train_cfg = Stage1ARunConfig(output_dir=OUTPUT)

if SMOKE:
    # Clearly labeled non-publication smoke test; not identical hyperparameters.
    train_cfg = replace(
        train_cfg,
        epochs=3,
        batch_size_per_slice=512
    )

if not SMOKE:
    assert train_cfg.epochs == 1500, train_cfg
    assert train_cfg.batch_size_per_slice == 1024, train_cfg
    assert train_cfg.seed == 20260622, train_cfg
    assert train_cfg.coordinate_decoder == "shared_auto", train_cfg
    assert train_cfg.gauge_mode == "global", train_cfg
    assert train_cfg.use_library_size_offset, train_cfg

# Verify source version's config is free of retired regularization arguments.
cfg_fields = set(asdict(train_cfg))
retired = {
    "submersion_weight", "scale_variation_weight",
    "coverage_prior_weight", "nuisance_weight", "slice_effect_dim"
}
if cfg_fields & retired:
    raise RuntimeError(
        f"Obsolete Stage-1a config fields: {cfg_fields & retired}"
    )

SEED = int(train_cfg.seed)
DEVICE = torch.device("cpu")
print("Spatial TI nine-section CPU training", flush=True)
print("  synthetic DATASET seed: 20261023 (only synthetic)", flush=True)
print("  training seed:", SEED, flush=True)
print("  git commit:", repo_commit, flush=True)
print("  CPU threads:", CPU_THREADS, flush=True)
print("  current method:", METHOD_VERSION, flush=True)
print("  default Stage1ARunConfig:", asdict(train_cfg), flush=True)
print("  output:", OUTPUT, flush=True)

# Freeze an explicit run identity; don't reuse incompatible previous outputs.
settings = {
    "experiment": "qian_pfc_spatial_ti_all9_matches_synthetic_stage1_defaults",
    "synthetic_dataset_seed_REFERENCE_ONLY": 20261023,
    "training_seed": SEED,
    "synthetic_reference_commit": PINNED_COMMIT,
    "active_commit": repo_commit,
    "method_version": METHOD_VERSION,
    "source_sha256": shasum(source_path),
    "device": "cpu",
    "cpu_threads": CPU_THREADS,
    "smoke": SMOKE,
    "dataset_root": str(DATA_ROOT),
    "count_source": "AnnData.raw.X",
    "targets": {
        "GW15": ["F1a", "F1b"],
        "GW20": ["F1", "F2a", "F2b"],
        "GW22": ["F1", "F2", "F3"]
    },
    "expected_panel": EXPECTED_PANEL,
    "training_config": {
        k: (str(v) if isinstance(v, Path) else v)
        for k, v in asdict(train_cfg).items()
    },
    "no_H1_or_layer_supervision": True,
}
OUTPUT.mkdir(parents=True, exist_ok=True)
if RUN_LOCK.exists():
    old = json.loads(RUN_LOCK.read_text())
    # Number of CPU threads is an execution detail, not a model change.
    old_cmp = {k: v for k, v in old.items() if k != "cpu_threads"}
    new_cmp = {k: v for k, v in settings.items() if k != "cpu_threads"}
    if old_cmp != new_cmp:
        raise RuntimeError(
            f"Existing run is incompatible with current settings:\n{RUN_LOCK}\n"
            "Use a NEW output directory; never overwrite an existing fit."
        )
elif any(OUTPUT.iterdir()):
    raise RuntimeError(
        f"Nonempty output directory lacks a lockfile: {OUTPUT}"
    )
else:
    RUN_LOCK.write_text(json.dumps(settings, indent=2) + "\n")

# -----------------------------------------------------------------------
# B. Load exact full nine-section PFC panel; no labels passed to training
# -----------------------------------------------------------------------
model_path = OUTPUT / "model_final.pt"

if FIT_DONE.is_file():
    saved = json.loads(FIT_DONE.read_text())
    assert saved["completed"] is True
    assert saved["commit"] == PINNED_COMMIT
    assert model_path.is_file()
    assert shasum(model_path) == saved["checkpoint_sha256"], (
        "A completed model file has changed. Refusing stale/falsified run."
    )
    print("EXISTING FINISHED FIT VERIFIED:", model_path, flush=True)
else:
    assert DATA_ROOT.is_dir(), DATA_ROOT
    for age in ("gw15", "gw20", "gw22"):
        assert (DATA_ROOT / f"{age}.h5ad").is_file()

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    targets = (
        ("GW15", ("F1a", "F1b")),
        ("GW20", ("F1", "F2a", "F2b")),
        ("GW22", ("F1", "F2", "F3"))
    )
    qcfg = QianPFCConfig(
        data_root=DATA_ROOT,
        targets=targets,
        expected_num_slices=9,
        layer_key_for_counts="raw",
        max_cells_per_slice=(3000 if SMOKE else None),
        max_genes_for_debug=None,
        split_multi_sample_regions=True,
        random_seed=SEED,
    )
    panel = build_qian_pfc_panel(qcfg).sort_values(
        "slice_id"
    ).reset_index(drop=True)
    actual = [
        (int(r["slice_id"]), str(r["gw"]), str(r["sample"]),
         str(r["region"]), int(r["n_full"]))
        for _, r in panel.iterrows()
    ]
    assert actual == EXPECTED_PANEL, (
        f"Unexpected panel:\n{actual}\nExpected:\n{EXPECTED_PANEL}"
    )
    print("\nExact nine-section cohort:")
    print(
        panel[
            ["slice_id", "gw", "sample", "region", "time", "n_full"]
        ].to_string(index=False),
        flush=True
    )

    loaded = load_qian_pfc_for_stage1(
        config=qcfg, panel_df=panel
    )
    counts = loaded.counts_by_slice
    spatial = loaded.spatial_by_slice
    size = [len(c) for c in counts]
    assert len(counts) == len(spatial) == 9
    assert len(loaded.gene_names) == 300
    assert len(loaded.obs_df) == sum(size)
    if not SMOKE:
        assert size == [int(r[4]) for r in EXPECTED_PANEL]
        assert sum(size) == 4_378_919

    # Full value audit (chunked) rather than just an integer-valued sample.
    for sid, c in enumerate(counts):
        for first in range(0, len(c), 65536):
            v = c[first:first + 65536]
            if not torch.is_tensor(v):
                v = torch.as_tensor(v)
            if not bool(torch.isfinite(v).all().item()):
                raise RuntimeError(f"Nonfinite counts, slice {sid}")
            if not bool((v >= 0).all().item()):
                raise RuntimeError(f"Negative raw counts, slice {sid}")
            if not bool(torch.allclose(v, v.round(), atol=1e-4, rtol=0)):
                raise RuntimeError(
                    f"Noninteger raw values in slice {sid}; "
                    "do not train Poisson count model on processed expression."
                )
        assert len(spatial[sid]) == len(c)
        print(
            f"  raw-count audit PASS: slice {sid}, {len(c):,} cells",
            flush=True
        )

    panel.to_csv(OUTPUT / "training_panel.csv", index=False)
    (OUTPUT / "gene_names.json").write_text(
        json.dumps(list(map(str, loaded.gene_names)), indent=2) + "\n"
    )

    # H1, layer, and other held-out annotations are NOT passed as labels.
    # Stage1a uses counts, coordinates, sample/age panel, and gene names.
    tic = time.time()
    result = run_stage1a(
        counts_by_slice=counts,
        spatial_by_slice=spatial,
        panel_df=loaded.panel_df,
        gene_names=loaded.gene_names,
        obs_df=None,
        config=train_cfg
    )
    print(f"Stage-1a fit finished after {(time.time()-tic)/60:.1f} minutes",
          flush=True)
    assert len(result.cells) == sum(size)
    assert np.isfinite(
        result.cells["z_stage1a"].to_numpy(dtype=float)
    ).all()

    # Stage1a writes its native model_final.pt and tables under OUTPUT.
    assert model_path.is_file(), (
        f"Trained model checkpoint not found at {model_path}"
    )
    FIT_DONE.write_text(json.dumps({
        "completed": True,
        "commit": PINNED_COMMIT,
        "training_seed": SEED,
        "method_version": METHOD_VERSION,
        "n_sections": 9,
        "n_cells": int(sum(size)),
        "n_genes": 300,
        "checkpoint_sha256": shasum(model_path),
    }, indent=2) + "\n")
    print("NEW FULL FIT VERIFIED:", model_path, flush=True)

# -----------------------------------------------------------------------
# C. No further training: derive native Fisher V(z,t), q(z,t) for QC.
# Same eps=1e-8 as synthetic; 128-step RK4 is used later for trajectories.
# -----------------------------------------------------------------------
if DYNAMICS_FILE.exists():
    print("Existing derived V/q grid:", DYNAMICS_FILE)
else:
    ckpt = torch.load(
        model_path, map_location="cpu", weights_only=False
    )
    assert {"model_config", "model_state_dict"} <= set(ckpt)
    m = FlowSTPModel(
        FlowSTPConfig(**ckpt["model_config"])
    ).to(DEVICE).eval()
    m.load_state_dict(ckpt["model_state_dict"], strict=True)
    for p in m.parameters():
        p.requires_grad_(False)

    z_grid = np.linspace(0, 1, 101, dtype=np.float32)
    t_grid = np.linspace(0, 1, 81, dtype=np.float32)
    Z, T = np.meshgrid(z_grid, t_grid, indexing="ij")
    velocity = np.empty(Z.size, dtype=np.float32)
    progression_rate = np.empty(Z.size, dtype=np.float32)
    tangent_energy = np.empty(Z.size, dtype=np.float32)

    with torch.no_grad():
        for first in range(0, Z.size, 2048):
            sl = slice(first, min(first + 2048, Z.size))
            z = torch.as_tensor(Z.ravel()[sl], device=DEVICE)
            t = torch.as_tensor(T.ravel()[sl], device=DEVICE)
            one, zero = torch.ones_like(z), torch.zeros_like(z)
            _, dz = torch.func.jvp(m.fisher_flat, (z, t), (one, zero))
            _, dt = torch.func.jvp(m.fisher_flat, (z, t), (zero, one))
            energy = dz.square().sum(dim=1)
            v = -(dz * dt).sum(dim=1) / (energy + 1e-8)
            q = torch.linalg.vector_norm(
                dt + v[:, None] * dz, dim=1
            )
            velocity[sl] = v.cpu().numpy()
            progression_rate[sl] = q.cpu().numpy()
            tangent_energy[sl] = energy.cpu().numpy()

    V = velocity.reshape(Z.shape)
    Q = progression_rate.reshape(Z.shape)
    TE = tangent_energy.reshape(Z.shape)
    if not all(np.isfinite(a).all() for a in (V, Q, TE)):
        raise FloatingPointError("Nonfinite Fisher dynamics; inspect checkpoint")
    if not np.all(Q >= 0):
        raise FloatingPointError("Negative Fisher residual speed")
    np.savez_compressed(
        DYNAMICS_FILE,
        z=z_grid, t=t_grid, V=V, q=Q, tangent_energy=TE,
        fisher_eps=np.asarray(1e-8),
        future_trajectory_rk4_steps=np.asarray(128),
    )
    print("Derived Fisher V/q grid saved:", DYNAMICS_FILE)

print("\nREADY: nine-section Spatial TI, synthetic Stage-1a defaults, CPU.")
print("Output:", OUTPUT)
print("Fitted field:", model_path)
print("Per-cell anatomical coordinates:", OUTPUT / "tables/obs_with_stage1a.csv.gz")
print("Derived Fisher dynamics:", DYNAMICS_FILE)
print("NOTE: Full per-cell RK4 paths are a SEPARATE inference step, not training.")

## Fit the strict five-section GW20 holdout model

In [ ]:
# Spatial TI: train on GW15 + GW22 only; hold out ALL GW20 sections.
# Same Stage-1a defaults as the latest full nine-tissue model.
# Run this entire cell on a CPU Slurm compute node (not the login node).

import os, sys, json, subprocess, inspect
from pathlib import Path

os.environ['CUDA_VISIBLE_DEVICES'] = ''
try:
    ncpu = len(os.sched_getaffinity(0))
except AttributeError:
    ncpu = os.cpu_count() or 4
ncpu = min(ncpu, int(os.environ.get('SLURM_CPUS_PER_TASK', ncpu)))
os.environ['OMP_NUM_THREADS'] = str(ncpu)
os.environ['MKL_NUM_THREADS'] = str(ncpu)

import numpy as np
import pandas as pd
import torch

torch.set_num_threads(ncpu)
try: torch.set_num_interop_threads(min(4, ncpu))
except RuntimeError: pass
if torch.cuda.is_available():
    raise RuntimeError('Use a fresh CPU Jupyter kernel for this run.')
if 'SLURM_JOB_ID' not in os.environ:
    print('WARNING: no Slurm allocation detected; do not train on a login node.')

PROJECT = Path(os.environ["STP_PROJECT"])
REPO = Path(os.environ.get('FLOW_STP_REPO',
    os.environ["FLOW_STP_REPO"]))
DATA_ROOT = Path(os.environ["QIAN_PFC_DATA_ROOT"])
OUT = PROJECT / 'experiments/qian_pfc_gw20_allheldout_gw15_gw22_five_tissue_v1/CPU_FULL'
PINNED_COMMIT = '76bb952a841c3671ec35161149e780167fd9b7b1'
if not (REPO / 'src/flow_stp/stage1/stage1a.py').is_file():
    if (PROJECT / 'src/flow_stp/stage1/stage1a.py').is_file(): REPO = PROJECT
    else: raise FileNotFoundError('Set FLOW_STP_REPO to the Flow-STP checkout')
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if head != PINNED_COMMIT:
    raise RuntimeError(f'Wrong Flow-STP commit: {head}; expected {PINNED_COMMIT}')
dirty = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain',
                                 '--untracked-files=no', '--', 'src/flow_stp'], text=True).strip()
if dirty: raise RuntimeError('Tracked Flow-STP source has local modifications: ' + dirty)
sys.path.insert(0, str(REPO / 'src'))
from flow_stp.data.qian_pfc import QianPFCConfig, build_qian_pfc_panel, load_qian_pfc_for_stage1
from flow_stp.stage1.stage1a import (Stage1ARunConfig, run_stage1a,
                                      METHOD_VERSION, FlowSTPModel, FlowSTPConfig)
if not Path(inspect.getfile(Stage1ARunConfig)).resolve().is_relative_to(REPO.resolve()):
    raise RuntimeError('Stale flow_stp import; restart the notebook kernel')
assert METHOD_VERSION == 'stage1a_shared_logE_global_gauge_v1'

# NO overrides to optimization hyperparameters.
cfg = Stage1ARunConfig(output_dir=OUT)
assert (cfg.epochs, cfg.batch_size_per_slice, cfg.learning_rate,
        cfg.weight_decay, cfg.seed, cfg.validation_fraction) == (
        1500, 1024, 2e-3, 1e-5, 20260622, .10)
assert cfg.coordinate_decoder == 'shared_auto' and cfg.gauge_mode == 'global'
assert cfg.use_library_size_offset

TARGETS = (('GW15', ('F1a', 'F1b')), ('GW22', ('F1', 'F2', 'F3')))
EXPECTED = [
    ('GW15', 'UMB1117', 'F1a', 548567),
    ('GW15', 'UMB1117', 'F1b', 410492),
    ('GW22', 'FB123', 'F1', 508417),
    ('GW22', 'FB123', 'F2', 439986),
    ('GW22', 'FB123', 'F3', 333113),
]
OUT.mkdir(parents=True, exist_ok=True)
if (OUT / 'FIT_VERIFIED.json').exists():
    proof = json.loads((OUT / 'FIT_VERIFIED.json').read_text())
    assert proof['train_ages'] == ['GW15', 'GW22'] and proof['n_slices'] == 5
    assert (OUT / 'model_final.pt').is_file()
    print('Verified completed held-out training:', OUT)
else:
    qcfg = QianPFCConfig(data_root=DATA_ROOT, targets=TARGETS,
                         expected_num_slices=5, layer_key_for_counts='raw',
                         split_multi_sample_regions=True, max_cells_per_slice=None,
                         max_genes_for_debug=None, random_seed=cfg.seed)
    panel = build_qian_pfc_panel(qcfg).sort_values('slice_id').reset_index(drop=True)
    actual = list(zip(panel.gw.astype(str), panel['sample'].astype(str),
                      panel.region.astype(str), panel.n_full.astype(int)))
    if actual != EXPECTED: raise RuntimeError(f'Unexpected panel {actual}; expected {EXPECTED}')
    assert all(x in ('GW15','GW22') for x in panel.gw)
    assert sum(panel.n_full) == 2240575
    panel['paper_label'] = ['GW15 Tissue 1','GW15 Tissue 2',
                            'GW22 Tissue 1','GW22 Tissue 2','GW22 Tissue 3']
    print('Training ONLY these tissues:\n', panel[['paper_label','n_full']].to_string(index=False))
    if (OUT / 'training_panel.csv').exists():
        earlier = pd.read_csv(OUT / 'training_panel.csv')
        if not earlier[['gw','sample','region','n_full']].equals(panel[['gw','sample','region','n_full']]):
            raise RuntimeError('Existing output was started with different tissues')
    else:
        if (OUT / 'model_final.pt').exists() or (OUT / 'checkpoint_last.pt').exists():
            raise RuntimeError('Found checkpoint without training_panel.csv; refusing to mix runs')
        panel.to_csv(OUT / 'training_panel.csv', index=False)
    data = load_qian_pfc_for_stage1(config=qcfg, panel_df=panel)
    assert len(data.counts_by_slice) == 5 and len(data.gene_names) == 300
    assert [len(x) for x in data.counts_by_slice] == [x[-1] for x in EXPECTED]
    for s, counts in enumerate(data.counts_by_slice):
        for j in range(0, len(counts), 65536):
            block = counts[j:j+65536]
            assert bool(torch.isfinite(block).all())
            assert bool((block >= 0).all())
            assert bool(torch.allclose(block, block.round(), atol=1e-4, rtol=0))
    (OUT / 'gene_names.json').write_text(json.dumps(data.gene_names, indent=2))
    # No annotations and no GW20 cells are supplied to training.
    result = run_stage1a(counts_by_slice=data.counts_by_slice,
                         spatial_by_slice=data.spatial_by_slice,
                         panel_df=data.panel_df, gene_names=data.gene_names,
                         obs_df=None, config=cfg)
    assert (OUT / 'model_final.pt').exists()
    assert (OUT / 'tables/stage1a_all_cells.csv.gz').exists()
    (OUT / 'FIT_VERIFIED.json').write_text(json.dumps({
        'completed': True, 'source_commit': head, 'n_slices': 5,
        'train_ages': ['GW15','GW22'], 'n_train_cells': 2240575,
        'num_genes': 300,
    }, indent=2))
    print('Training completed:', OUT / 'model_final.pt')

# Freeze the age-20 expression field E(z,t), without reading GW20 data.
FORECAST = OUT / 'GW20_FROZEN_expression_field_zgrid.npz'
if not FORECAST.exists():
    saved = torch.load(OUT / 'model_final.pt', map_location='cpu', weights_only=False)
    model = FlowSTPModel(FlowSTPConfig(**saved['model_config'])).eval()
    model.load_state_dict(saved['model_state_dict'], strict=True)
    assert model.config.num_slices == 5
    assert model.config.coordinate_decoder == 'shared_time_tissue'
    z = np.linspace(0, 1, 201, dtype=np.float32)
    t = np.full_like(z, 5/7, dtype=np.float32)
    with torch.inference_mode():
        rates = model.expression(torch.from_numpy(z), torch.from_numpy(t)).cpu().numpy()
    assert rates.shape == (201,300) and np.isfinite(rates).all()
    gene_names = json.loads((OUT/'gene_names.json').read_text())
    np.savez_compressed(FORECAST, z=z, age_gw=20, t_normalized=5/7,
                        gene_names=np.array(gene_names, dtype=str),
                        expression_rate=rates,
                        gene_proportion=rates/rates.sum(axis=1, keepdims=True))
print('GW20 held out completely; model:', OUT / 'model_final.pt')
print('Frozen age-20 prediction:', FORECAST)
print('NOTE: Per-cell GW20 prediction needs a separate leakage-free z assignment.')